# Descoberta das fontes - Base dos Dados / BigQuery

Objetivo: identificar os `dataset_id`/`table_id` reais das 6 fontes exigidas pelo desafio
(UF, Município, Meta Alfabetização Brasil/UF/Município, Dados de alunos) no datalake
público da Base dos Dados, antes de escrever o script de ingestão da camada Bronze.

In [1]:
import basedosdados as bd

bd.config.billing_project_id = "brazil-literacy-lakehouse"

In [4]:
import pandas as pd

resultados = bd.search(q="criança alfabetizada")
pd.DataFrame(resultados)

,slug,name,description,n_tables,n_raw_data_sources,n_information_requests,organization
0,br_inep_avaliacao_alfabetizacao,Avaliação da Alfabetização,O Compromisso Nacional Criança Alfabetizada é ...,6,1,0,"{'slug': 'inep', 'name': 'Instituto Nacional d..."
1,learning_poverty,Learning Poverty Global Database,None,1,0,0,"{'slug': 'wb', 'name': 'Banco Mundial'}"
2,pirls,Progress in International Reading Literacy Stu...,The IEA's Progress in International Reading Li...,7,1,0,"{'slug': 'iea', 'name': 'International Associa..."
3,censo_2022,Censo 2022,Conhecer em detalhe como é e como vive o nosso...,15,1,0,"{'slug': 'ibge', 'name': 'Instituto Brasileiro..."
4,open_africa,openAFRICA,openAFRICA aims to be the largest independent ...,0,1,0,"{'slug': 'code_for_africa', 'name': 'Code for ..."
5,peacekeeping_master_open_datasets,Peacekeeping Master Open Datasets,Peacekeeping Master Open Datasets são listas q...,0,1,0,"{'slug': 'onu', 'name': 'Organização das Naçõe..."


In [6]:
query = """
SELECT table_name
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
"""

df_tabelas = bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")
df_tabelas

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform&state=P3pr2WDwMKReKgmien9dvQvtCWNZPJ&code_challenge=2r9a-9VlPlY53TL7MJwzxOdlm1q9XXeFSAWBCHjCxKc&code_challenge_method=S256&prompt=consent&access_type=offline
Downloading: 100%|██████████|


,table_name
0,dicionario
1,meta_alfabetizacao_brasil
2,alunos
3,municipio
4,meta_alfabetizacao_municipio
5,meta_alfabetizacao_uf
6,uf


In [7]:
DATASET_ID = "br_inep_avaliacao_alfabetizacao"

FONTES = {
    "uf": "uf",
    "municipio": "municipio",
    "meta_alfabetizacao_brasil": "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": "meta_alfabetizacao_municipio",
    "alunos": "alunos",
}

In [8]:
query = f"""
SELECT *
FROM `basedosdados.{DATASET_ID}.uf`
LIMIT 5
"""
bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

Downloading: 100%|██████████|


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
query = f"""
SELECT
  COUNT(*) AS total_linhas,
  COUNTIF(proporcao_aluno_nivel_0 IS NOT NULL) AS preenchidos_nivel_0,
  COUNTIF(taxa_alfabetizacao IS NOT NULL) AS preenchidos_taxa,
  COUNT(DISTINCT ano) AS anos_distintos,
  COUNT(DISTINCT rede) AS redes_distintas
FROM `basedosdados.{DATASET_ID}.uf`
"""
bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

Downloading: 100%|██████████|


,total_linhas,preenchidos_nivel_0,preenchidos_taxa,anos_distintos,redes_distintas
0,145,75,145,2,4


In [ ]:
for nome, tabela in FONTES.items():
    if nome == "uf":
        continue 
    query = f"SELECT COUNT(*) AS total_linhas FROM `basedosdados.{DATASET_ID}.{tabela}`"
    resultado = bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")
    print(f"{nome} ({tabela}): {resultado['total_linhas'][0]:,} linhas")

Downloading: 100%|██████████|
municipio (municipio): 23,995 linhas
Downloading: 100%|██████████|
meta_alfabetizacao_brasil (meta_alfabetizacao_brasil): 3 linhas
Downloading: 100%|██████████|
meta_alfabetizacao_uf (meta_alfabetizacao_uf): 81 linhas
Downloading: 100%|██████████|
meta_alfabetizacao_municipio (meta_alfabetizacao_municipio): 10,704 linhas
Downloading: 100%|██████████|
alunos (alunos): 3,867,999 linhas


## Conclusões da descoberta

- Dataset confirmado: `basedosdados.br_inep_avaliacao_alfabetizacao`
- As 6 fontes exigidas mapeadas: uf, municipio, meta_alfabetizacao_brasil,
  meta_alfabetizacao_uf, meta_alfabetizacao_municipio, alunos
- Existe uma 7ª tabela, `dicionario`, útil pra decodificar valores na Silver
- API de metadados da BD (`bd.search`/`bd.get_tables`) apresentou instabilidade;
  usar `INFORMATION_SCHEMA.TABLES` direto no BigQuery é mais confiável
- Achado de qualidade: em `uf`, `proporcao_aluno_nivel_0` só ~52% preenchida,
  investigar o padrão exato na Fase 4 (Silver)